# Stage 4, chunk 7: evaluate the chat-tuned model

1. **Chat behavior** of the base model, the Stage 3 model and the Stage 4 model: the hand-written prompt set (60 prompts), thinking on and off, greedy, each model served by vLLM in turn. This also gives the base and Stage 3 baselines.
2. **Quality**, the Stage 3 protocol: bits per character on held-out Urdu, FLORES Urdu and English, and Belebele, for the Stage 3 and Stage 4 models.
3. **The success checks** of `docs/stage4-plan.md` as PASS or FAIL.

About 1.5 to 2 hours. Settings before running:

- Accelerator: **GPU T4 x2**. Internet: **on**. No secret.
- Inputs (**Add Input > Notebooks**): the output of `07_adapt_main_qwen3` and of `14_tune_sft`.
- Run with **Save Version > Save & Run All (Commit)**, then send back `tune_eval.zip` (Output tab).

In [ ]:
# Install zubaan at one exact commit. Do not edit the next line except to change the commit.
COMMIT = "2e0634e"
REPO = f"git+https://github.com/Hashir-Aziz123/zubaan.git@{COMMIT}"
!pip install -q "zubaan[audit,train] @ {REPO}"
# Kaggle may already hold an older copy with the same version number, which pip keeps.
# Force our commit (dependencies are already in place).
!pip install -q --force-reinstall --no-deps "zubaan @ {REPO}"
# Kaggle ships torchao 0.10, which peft refuses ("only versions above 0.16 are supported") when it
# builds LoRA layers. We never use torchao, so remove it; peft then skips its torchao support.
!pip uninstall -y -q torchao
!zubaan version

In [ ]:
# CHECK 1: the installed code is the commit above, not a cached older copy.
# If this fails, use Run > Restart session and run the cells again from the top.
import json
from importlib import metadata

direct = json.loads(metadata.distribution("zubaan").read_text("direct_url.json"))
installed = direct["vcs_info"]["commit_id"]
print("installed commit:", installed)
assert installed.startswith(COMMIT), f"installed {installed}, wanted {COMMIT}"

import zubaan.adapt.pipeline  # noqa: F401

print("OK: code version verified")

In [ ]:
import os
import subprocess

# Quiet progress bars: they can overwhelm the notebook front end.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# CHECK 2: a GPU with about 16 GB. Training uses one GPU (fp16: a T4 and a P100 have no BF16).
try:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
except FileNotFoundError:
    raise SystemExit("No GPU in this session (nvidia-smi is missing). Settings > Accelerator > GPU T4 x2, then run again.") from None
assert r.returncode == 0, "no GPU found: turn on a GPU accelerator in the notebook settings"
print(r.stdout)
gpu_name, cap, mem = [x.strip() for x in r.stdout.splitlines()[0].split(",")]
assert float(mem.split()[0]) >= 14000, f"{gpu_name} has {mem}; need about 16 GB"
if float(cap) < 7.0:
    print("WARNING: compute capability", cap, "has no fp16 tensor cores; this will be slow. A T4 is better.")

import peft
import torch
import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__, "| cuda", torch.version.cuda)
assert torch.cuda.is_available(), "torch cannot see the GPU"
print("OK: GPU", gpu_name, "compute capability", cap, mem)

# Both teachers are split over the two T4s (tensor parallel 2): Qwen3-8B in float16 is about 16 GB
# of weights and does not fit one 16 GB T4.
gpus = [x for x in r.stdout.splitlines() if x.strip()]
assert len(gpus) == 2, f"found {len(gpus)} GPU(s); pick GPU T4 x2 in the notebook settings"
print("OK: two GPUs")

In [ ]:
from pathlib import Path

OUT = Path("/kaggle/working/tune_eval")
OUT.mkdir(parents=True, exist_ok=True)
MAX_MODEL_LEN = 4096   # prompt plus the longest reply the check allows (2,048 with thinking)
WORKERS = 16           # chat-check requests in flight; vLLM batches them

## 1. Models

In [ ]:
# Inputs: the output of notebook 07 (Stage 3 model) and of notebook 14 (Stage 4 model). The base
# model is downloaded from the Hub.
INPUT_ROOT = Path("/kaggle/input")
stage3, stage4 = [], []
for m in sorted(INPUT_ROOT.rglob("zubaan_extension.json")):
    meta = json.loads(m.read_text(encoding="utf-8"))
    if meta.get("trained"):
        (stage4 if "stage4" in meta else stage3).append(m.parent)
assert len(stage3) == 1 and len(stage4) == 1, f"need one Stage 3 and one Stage 4 model, found {stage3} and {stage4}"
MODELS = {"base": "Qwen/Qwen3-1.7B", "stage3": str(stage3[0]), "stage4": str(stage4[0])}
a = json.loads((stage3[0] / "tokenizer.json").read_text(encoding="utf-8"))
b = json.loads((stage4[0] / "tokenizer.json").read_text(encoding="utf-8"))
assert a == b, "Stage 3 and Stage 4 tokenizers differ: SFT must not touch the tokenizer"
print(json.dumps(MODELS, indent=1))

## 2. Chat check (vLLM, isolated environment)

In [ ]:
# vLLM goes into its own Python 3.12 environment so it cannot replace Kaggle's torch. Takes 5 to 10 minutes.
!pip install -q uv
!uv venv --python 3.12 /tmp/vllm-env
!uv pip install --python /tmp/vllm-env/bin/python vllm
VLLM_VERSION = subprocess.run(
    ["/tmp/vllm-env/bin/python", "-c", "import vllm; print(vllm.__version__)"], capture_output=True, text=True
).stdout.strip()
print("vLLM", VLLM_VERSION)
assert VLLM_VERSION, "vLLM did not install"

In [ ]:
import os
import signal
import time
import urllib.request

from zubaan.tune.generate import teacher_serve_command

ENV = dict(os.environ, PATH="/tmp/vllm-env/bin:" + os.environ["PATH"])


def gpu_used_mb():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
    return [int(x) for x in out.split()]


def start_teacher(model, served):
    log = OUT / f"vllm_{served}.log"
    cmd = teacher_serve_command(model, served, tensor_parallel=2, max_model_len=MAX_MODEL_LEN)
    print(" ".join(cmd))
    with open(log, "w") as log_file:  # the server keeps its own copy of the handle
        proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, env=ENV, start_new_session=True)
    for _ in range(240):  # up to 40 minutes: download, load, CUDA graph capture
        time.sleep(10)
        if proc.poll() is not None:
            break
        try:
            with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=10) as r:
                if served in r.read().decode():
                    print("server up:", served)
                    return proc
        except Exception:
            pass
    print(log.read_text()[-4000:])
    raise RuntimeError(f"vLLM did not start for {model}: see {log}")


def stop_teacher(proc):
    os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
    for _ in range(60):
        time.sleep(5)
        if proc.poll() is not None and max(gpu_used_mb()) < 1000:
            break
    print("stopped; GPU memory in use (MB):", gpu_used_mb())

In [ ]:
# The same hand-written prompt set, greedy decoding and limits for all three models, each served by
# vLLM in turn. Replies are saved as they arrive; a rerun after a crash resumes each model.
from zubaan.config import ChatCheckConfig
from zubaan.tune.chat_check import run_from_config

chat = {}
for label, model in MODELS.items():
    out_file = OUT / f"chat_check_{label}.json"
    if out_file.exists():
        chat[label] = json.loads(out_file.read_text(encoding="utf-8"))["summary"]
        print(label, "already done")
        continue
    proc = start_teacher(model, label)
    try:
        cfg = ChatCheckConfig(label=label, server_url="http://127.0.0.1:8000", model=label, workers=WORKERS, output=out_file)
        chat[label] = run_from_config(cfg, log=print)["summary"]
    finally:
        stop_teacher(proc)
    for mode, groups in chat[label].items():
        g = groups["all"]
        print(f"{label} {mode}: answered {g['answered']}/{g['n']}, hit limit {g['hit_limit']}, loops {g['loops']}")

## 3. Quality (Stage 3 protocol)

In [ ]:
import yaml

# The Stage 3 protocol (notebook 08): 1000 texts of at most 400 characters per set, all 900 Belebele
# questions per language, float32. The base model is not re-scored (its numbers are recorded); the
# Stage 3 model is, to show the protocol still reproduces notebook 08.
evaluate = {
    "models": {"stage3": MODELS["stage3"], "stage4": MODELS["stage4"]},
    "untrained": None,
    "adapted": None,
    "heldout": "data/urdu/urdu_heldout.txt",
    "eval_urdu": "data/flores_urd.txt",
    "eval_english": "data/flores_eng.txt",
    "quality_texts": 1000,
    "max_chars": 400,
    "belebele_questions": None,
    "dtype": "float32",
    "device": "cuda",
    "batch_size": 4,
    "max_batch_tokens": 2048,
    "output": str(OUT / "tune_quality.json"),
}
config = {
    "urdu_corpus": {"raw_dir": "data/raw", "out_dir": "data/urdu", "flores": "data/flores_urd.txt", "max_mb": 100, "heldout_percent": 1.0},
    "evaluate": evaluate,
}
Path("evaluate.yaml").write_text(yaml.safe_dump(config, sort_keys=False))
print(Path("evaluate.yaml").read_text())

In [ ]:
# The same evaluation text as Stage 2: FLORES devtest, and the held-out Urdu Wikipedia lines.
!zubaan data flores --out data
!zubaan data urdu --config evaluate.yaml

In [ ]:
meta = json.loads(Path("data/urdu/urdu_meta.json").read_text())
sha = meta["files"]["heldout"]["sha256"]
print("held-out sha256:", sha)
# Stage 2 measured the base model on a file with this hash. A different file would make the rows
# incomparable with the Stage 2 table, so stop.
assert sha == "19c4f1b63b41b8a91cf8aa345d0f0e83945f33e403cc78c1f6f52ed7e7c34ada", "the held-out Urdu file differs from the one Stage 2 used"
print("OK: same held-out file as Stage 2")

In [ ]:
# About 25 minutes per model on a T4. Finished models are skipped on a rerun.
!zubaan adapt eval --config evaluate.yaml

## 4. Success checks

In [ ]:
import shutil

from zubaan.tune.evaluate import checks_markdown, reproduces_stage3, stage4_checks

quality = json.loads((OUT / "tune_quality.json").read_text(encoding="utf-8"))["variants"]
drift = reproduces_stage3(quality["stage3"])
assert not drift, f"the Stage 3 model no longer reproduces notebook 08, so the protocol drifted: {drift}"
print("OK: the Stage 3 model reproduces its notebook 08 numbers")
checks = stage4_checks(chat, quality)
md_text = checks_markdown(checks, chat)
(OUT / "stage4_checks.json").write_text(json.dumps({"checks": checks, "chat": chat, "vllm": VLLM_VERSION, "gpu": gpu_name}, indent=1, ensure_ascii=False))
(OUT / "stage4_checks.md").write_text(md_text, encoding="utf-8")
print(md_text)
shutil.make_archive("/kaggle/working/tune_eval", "zip", OUT)
print("download /kaggle/working/tune_eval.zip from the Output tab")